# Reasoning Strategies for Agents
Companion notebook for the **Reasoning Strategies for Agents** section of Chapter 8.

Use the **smallest reasoning pattern that solves the task well**. Tool-first when correctness depends on fresh or exact external facts; model-first for transformation, summarization, or drafting from context you already have.

A small `Task` shape so the book's `choose_reasoning` function has something concrete to inspect:

In [1]:
from dataclasses import dataclass


@dataclass
class Task:
    needs_fresh_data: bool = False
    steps: int = 1
    has_dependencies: bool = False
    is_dynamic: bool = False

This is the book's `choose_reasoning`, unchanged:

In [2]:
def choose_reasoning(task):
    if task.needs_fresh_data:
        return "tool_first"
    if task.steps > 3 or task.has_dependencies:
        return "plan_then_execute"
    if task.is_dynamic:
        return "react"
    return "direct"

In [3]:
examples = {
    "check order status": Task(needs_fresh_data=True),
    "review case, fetch account, check policy, draft reply": Task(steps=4, has_dependencies=True),
    "answer follows up on unpredictable tool output": Task(is_dynamic=True),
    "reformat this paragraph": Task(),
}

for description, task in examples.items():
    print(f"{choose_reasoning(task):18s} <- {description}")

tool_first         <- check order status
plan_then_execute  <- review case, fetch account, check policy, draft reply
react              <- answer follows up on unpredictable tool output
direct             <- reformat this paragraph


**Plan-before-act** suits tasks with many steps or dependencies -- write a short checklist, then execute it, which keeps tool use from wandering. **ReAct** (reason, call a tool, observe, continue) suits dynamic work where the next step depends on what a tool just returned. Don't reach for either on simple classification, formatting, or fixed routing -- that's what `"direct"` is for.